In [0]:
from openai import OpenAI
import os

# How to get your Databricks token: https://docs.databricks.com/en/dev-tools/auth/pat.html
# DATABRICKS_TOKEN = os.environ.get('DATABRICKS_TOKEN')
# Alternatively in a Databricks notebook you can use this:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url="https://7474648169521809.ai-gateway.cloud.databricks.com/mlflow/v1"
)

response = client.chat.completions.create(
    model="databricks-gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "I would like to build an agent. can you give me suggestions"
        }
    ],
    max_tokens=5000
)
a = response.choices[0].message.content
print(response.choices[0].message.content)

In [0]:
type(a)

In [0]:
a[0]

In [0]:
a[1]

In [0]:
type(a[1])

In [0]:
a[1].keys()

In [0]:
a[1].get("text")


'Below is a practical, step‑by‑step checklist you can follow when building an autonomous (AI) agent. Feel free to pick and choose the items that fit your use‑case and technology stack.\n\n---\n\n## 1. Clarify the Agent’s Goal & Scope\n| Question | Why it matters |\n|----------|----------------|\n| **What problem am I solving?** | Guides data, tools, and performance metrics. |\n| **Is the agent conversational, task‑oriented, or both?** | Determines the need for dialogue management vs. pure automation. |\n| **What are the success criteria?** (e.g., accuracy, latency, cost) | Enables objective evaluation later. |\n| **What constraints exist?** (privacy, latency, offline, budget) | Influences model size, hosting, and tool choices. |\n\n**Output:** A short “agent charter” (1–2 sentences) plus a list of measurable KPIs.\n\n---\n\n## 2. Choose a Core Architecture\n\n| Component | Typical Options | Quick Decision Guide |\n|-----------|----------------|----------------------|\n| **LLM (brain)** | • OpenAI GPT‑4o / GPT‑4‑turbo  <br>• Anthropic Claude 3 <br>• Llama‑3 / Mistral (self‑hosted) | If you need top‑tier reasoning and can afford API calls → OpenAI/Anthropic. <br>If you need on‑prem or low cost → fine‑tuned Llama‑3. |\n| **Prompting style** | • Zero‑shot / few‑shot <br>• ReAct (reason‑act loop) <br>• Chain‑of‑thought | For autonomous agents, **ReAct** + **tool use** prompting works best. |\n| **Tool‑use framework** | • LangChain / LlamaIndex (Python) <br>• CrewAI (task‑orchestration) <br>• Auto‑GPT / Agentic‑GPT (out‑of‑the‑box) | If you want modularity and fine control → LangChain. <br>If you need quick prototyping → Auto‑GPT. |\n| **Memory** | • Short‑term (conversation buffer) <br>• Long‑term (vector DB: Pinecone, Chroma, Weaviate) | Use short‑term for context windows; long‑term only if the agent must recall past interactions or knowledge over days. |\n| **Execution environment** | • Cloud Functions (AWS Lambda, GCP Cloud Run) <br>• Container (Docker + Kubernetes) <br>• Local process | Start with a simple serverless function; scale to containers when you need concurrency or GPU. |\n\n---\n\n## 3. Draft the Agent’s Workflow\n\n```\nUser Input → (Pre‑processing) → LLM Prompt → (LLM decides) → \n   ├─► Use Tool A (e.g., web search) → Return data\n   ├─► Use Tool B (e.g., database query) → Return data\n   └─► Continue reasoning → Final response\n```\n\n**Key design points**\n\n1. **Tool Registry** – Define a JSON/YAML file that maps tool names → function signatures (input schema, output schema).  \n2. **Decision Loop** – Implement a ReAct loop (`Thought → Action → Observation → Thought …`) until the LLM outputs `final_answer`.  \n3. **Safety Guardrails** – Insert a moderation check after each LLM output (OpenAI moderation endpoint, custom regex, or safety model).  \n4. **Error Handling** – Retry on tool failures, fall back to a “cannot complete” message after N attempts.\n\n---\n\n## 4. Build a Minimal Viable Agent (MVP)\n\n1. **Set up a sandbox** (Python virtualenv).  \n2. **Install core libraries**  \n\n```bash\npip install openai langchain[all] chromadb\n```\n\n3. **Create a simple tool** (e.g., Wikipedia search)  \n\n```python\nfrom langchain.tools import BaseTool\nimport requests\n\nclass WikiSearch(BaseTool):\n    name = "wiki_search"\n    description = "Search Wikipedia for a short summary."\n\n    def _run(self, query: str):\n        r = requests.get(\n            "https://en.wikipedia.org/api/rest_v1/page/summary/" + query.replace(" ", "_")\n        )\n        return r.json().get("extract", "No result")\n```\n\n4. **Wire the agent**  \n\n```python\nfrom langchain.chat_models import ChatOpenAI\nfrom langchain.agents import initialize_agent, AgentType\n\nllm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)\ntools = [WikiSearch()]\n\nagent = initialize_agent(\n    tools,\n    llm,\n    agent=AgentType.REACT,          # ReAct prompting\n    verbose=True,\n    handle_parsing_errors=True,\n)\n\nprint(agent.run("Give me a quick overview of quantum teleportation."))\n```\n\n5. **Test** with several prompts, note token usage and latency, then iterate.\n\n---\n\n## 5. Expand & Harden\n\n| Area | What to add | Why |\n|------|-------------|-----|\n| **Additional tools** | • Web search (SerpAPI) <br>• Calculator <br>• Database API <br>• File read/write | Increases autonomy. |\n| **Long‑term memory** | Store past interactions in a vector DB (e.g., Chroma). Retrieve relevant chunks before each prompt. | Helps with multi‑session agents. |\n| **User authentication** | OAuth tokens, API keys. | Prevents abuse & personalizes responses. |\n| **Rate limiting & cost monitoring** | Wrap API calls with a decorator that logs tokens & cost. | Keeps budget under control. |\n| **Observability** | Structured logs, tracing (OpenTelemetry), health endpoints. | Easier debugging in production. |\n| **Safety** | • Prompt injection filtering <br>• Output red‑team testing <br>• Allow‑list of permissible actions. | Reduces hallucinations & malicious behavior. |\n| **Deployment** | • Dockerize the app <br>• Deploy to Cloud Run / ECS / Azure Functions <br>• Set up CI/CD (GitHub Actions). | Enables scaling & reliability. |\n| **Evaluation** | • Automated test suite (prompt → expected JSON) <br>• Human‑in‑the‑loop rating (MAUDE, helpfulness). | Measure progress against the KPIs you defined. |\n\n---\n\n## 6. Recommended Resources & Tools\n\n| Category | Resource |\n|----------|----------|\n| **Core LLM APIs** | OpenAI (gpt‑4o, gpt‑4‑turbo), Anthropic, Cohere |\n| **Agent Frameworks** | LangChain, LlamaIndex, CrewAI, Auto‑GPT, Agentic‑GPT |\n| **Tool Integration** | SerpAPI (search), Tavily (open‑source search), PDFPlumber (document parsing) |\n| **Vector Stores** | Chroma, Pinecone, Weaviate, Qdrant |\n| **Safety** | OpenAI Moderation, Anthropic Guardrails, “PromptGuard” library |\n| **Tutorials** | LangChain “Agents” docs, OpenAI “Function calling” guide, “Build an Auto‑GPT” blog series |\n| **Community** | /r/LocalLLM, LangChain Discord, AI Agents Slack, Papers with Code (search “agentic LLM”) |\n\n---\n\n## 7. Quick Checklist Before Going Live\n\n- [ ] **Goal & KPI** clearly documented.  \n- [ ] **Prompt template** stable (few‑shot examples, tool descriptions).  \n- [ ] **All tools** have robust error handling & timeout.  \n- [ ] **Safety layer** (moderation + prohibited‑action whitelist).  \n- [ ] **Observability** (logs, metrics, health check).  \n- [ ] **Cost guardrails** (max tokens per request, budget alerts).  \n- [ ] **Version control** (Git repo with CI that runs unit tests).  \n- [ ] **Rollback plan** (docker image tag, feature flag).  \n\n---\n\n### TL;DR\n\n1. **Define a crystal‑clear purpose** → KPIs.  \n2. **Pick an LLM** (OpenAI, Anthropic, or self‑hosted) and a **prompting style** (ReAct works well).  \n3. **Wrap external capabilities as tools** (search, DB, calculator).  \n4. **Use an agent framework** (LangChain is the most flexible) to orchestrate the Thought‑Action‑Observation loop.  \n5. **Implement memory, safety, and observability** early; they’re cheaper than retro‑fitting them.  \n6. **Iterate from a minimal prototype** → add tools, long‑term memory, authentication, deployment.  \n7. **Monitor cost, latency, and quality**; adjust prompts or model size as needed.\n\nGood luck building your agent! If you have a specific domain (e.g., finance, customer support, data analysis) let me know and I can suggest domain‑specific tools and prompt patterns.'